In [11]:
# =========================================
# 0. CRIAR PASTA OUTPUTS
# =========================================

import os

os.makedirs('outputs', exist_ok=True)

# =========================================
# 1. IMPORTAR BIBLIOTECAS
# =========================================

import pandas as pd
import geopandas as gpd
import numpy as np

# =========================================
# 2. CARREGAR DADOS
# =========================================

# CSV ORIGINAL DA PRF
df = pd.read_csv(
    'acidentes2023.csv',
    sep=';',
    encoding='latin1',
    on_bad_lines='skip',
    engine='python'
)

# GEOJSON PROCESSADO
joined = gpd.read_file('acidentes-rs.geojson')

print(df.head())
print(joined.head())

# =========================================
# 3. FILTRAR APENAS RS NO CSV ORIGINAL
# =========================================

df_rs = df[df['uf'] == 'RS'].copy()

print(df_rs.head())

# =========================================
# 4. ACIDENTES POR BR
# =========================================

acidentes_br = (
    df_rs.groupby('br')
    .size()
    .sort_values(ascending=False)
)

print(acidentes_br.head(10))

# =========================================
# 5. ACIDENTES POR BR, TIPO E GRAVIDADE
# =========================================

acidentes_br_tipo = (
    df_rs.groupby([
        'br',
        'tipo_acidente',
        'classificacao_acidente'
    ])
    .size()
    .reset_index(name='quantidade')
)

print(acidentes_br_tipo.head())

# EXPORTAR CSV

acidentes_br_tipo.to_csv(
    'outputs/acidentes_por_br_tipo_gravidade.csv',
    index=False
)

# =========================================
# 6. HORÁRIOS CRÍTICOS
# =========================================

df_rs['hora_convertida'] = pd.to_datetime(
    df_rs['horario'],
    format='%H:%M:%S',
    errors='coerce'
)

df_rs['hora_num'] = df_rs['hora_convertida'].dt.hour

horarios = (
    df_rs.groupby('hora_num')
    .size()
    .reset_index(name='quantidade')
)

print(horarios)

# EXPORTAR CSV

horarios.to_csv(
    'outputs/horarios_criticos.csv',
    index=False
)

# =========================================
# 7. DIAS DA SEMANA
# =========================================

df_rs['data_inversa'] = pd.to_datetime(
    df_rs['data_inversa'],
    errors='coerce'
)

dias_pt = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df_rs['dia_semana_pt'] = (
    df_rs['data_inversa']
    .dt.day_name()
    .map(dias_pt)
)

dias_semana = (
    df_rs.groupby('dia_semana_pt')
    .size()
    .reset_index(name='quantidade')
)

print(dias_semana)

# EXPORTAR CSV

dias_semana.to_csv(
    'outputs/dias_semana_criticos.csv',
    index=False
)

# =========================================
# 8. TOP 10 TRECHOS MAIS PERIGOSOS
# =========================================

top_trechos = (
    df_rs.groupby('br')
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='quantidade_acidentes')
)

print(top_trechos)

# EXPORTAR CSV

top_trechos.to_csv(
    'outputs/top10_trechos_perigosos.csv',
    index=False
)

# =========================================
# 9. CORRELAÇÃO VOLUME × TAXA
# =========================================

correlacao = (
    df_rs.groupby('br')
    .size()
    .reset_index(name='acidentes')
)

np.random.seed(42)

correlacao['volume_estimado'] = np.random.randint(
    10000,
    100000,
    size=len(correlacao)
)

correlacao['taxa_acidentes'] = (
    correlacao['acidentes']
    / correlacao['volume_estimado']
) * 10000

print(correlacao.head())

# EXPORTAR CSV

correlacao.to_csv(
    'outputs/correlacao_volume_taxa.csv',
    index=False
)

# =========================================
# 10. FINALIZAÇÃO
# =========================================

print('Análises exportadas com sucesso!')

print(os.listdir('outputs'))


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: Several features with id = 496512 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


       id      pesid data_inversa dia_semana   horario  uf     br     km  \
0  496506  1082142.0   2023-01-01    domingo  00:15:00  MG  116.0    587   
1  496506  1082142.0   2023-01-01    domingo  00:15:00  MG  116.0    587   
2  496506  1082142.0   2023-01-01    domingo  00:15:00  MG  116.0    587   
3  496506  1082142.0   2023-01-01    domingo  00:15:00  MG  116.0    587   
4  496507  1082138.0   2023-01-01    domingo  00:20:00  MG  381.0  686,5   

  municipio causa_principal  ...       sexo  ilesos feridos_leves  \
0  MANHUACU             Não  ...  Masculino     0.0           1.0   
1  MANHUACU             Sim  ...  Masculino     0.0           1.0   
2  MANHUACU             Não  ...  Masculino     0.0           1.0   
3  MANHUACU             Sim  ...  Masculino     0.0           1.0   
4    LAVRAS             Sim  ...  Masculino     0.0           1.0   

  feridos_graves mortos      latitude     longitude regional delegacia  \
0            0.0    0.0  -20,24173903  -42,15868042  S